Import Necessary Libraries

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import re
from tqdm import tqdm
import requests

GITHUB Details - Replace GITHUB ACCESS KEY with your own Personal access token

In [ ]:
github_token = "GITHUB ACCESS KEY"
headers = {"Authorization": f"token {github_token}"}

Discussion to Issues

In [2]:
df = pd.read_csv("../Dataset/DiscussionToIssue.csv")

In [3]:
encoder = LabelEncoder()
df['IsIssueRaised'] = encoder.fit_transform(df['IsIssueRaised'])
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [4]:
df_shuffled['concatenated'] = df_shuffled['Title'] + ' ' + df_shuffled['Description'] + ' ' + df_shuffled['Comments']

Training ...

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df_shuffled['concatenated'], df['IsIssueRaised'], test_size=0.2, random_state=42
)

In [6]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [7]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5315315315315315

Classification Report:
               precision    recall  f1-score   support

           0       0.45      0.24      0.31       197
           1       0.56      0.77      0.65       247

    accuracy                           0.53       444
   macro avg       0.50      0.50      0.48       444
weighted avg       0.51      0.53      0.50       444



Discussion to Issues with Description alone

In [8]:
df_shuffled['concatenated'] = df_shuffled['Title'] + ' ' + df_shuffled['Description']

Training ...

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df_shuffled['concatenated'], df['IsIssueRaised'], test_size=0.2, random_state=42
)

In [10]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [11]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.49099099099099097

Classification Report:
               precision    recall  f1-score   support

           0       0.37      0.22      0.28       197
           1       0.53      0.71      0.61       247

    accuracy                           0.49       444
   macro avg       0.45      0.46      0.44       444
weighted avg       0.46      0.49      0.46       444



Discussion to Issues with First Comment Alone

Function to extract Repository and Discussion number

In [12]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the First Comment of the Discussion

In [ ]:
df_shuffled['Comment'] =  None
for index,row in tqdm(df_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df_shuffled.at[index, 'Comment'] = repo_comment

2218it [16:33,  2.23it/s]


In [15]:
df_shuffled['concatenated'] = df_shuffled['Title'] + ' ' + df_shuffled['Description']+' '+str(df_shuffled['Comment'])

Training ...

In [16]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [17]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.49099099099099097

Classification Report:
               precision    recall  f1-score   support

           0       0.37      0.22      0.28       197
           1       0.53      0.71      0.61       247

    accuracy                           0.49       444
   macro avg       0.45      0.46      0.44       444
weighted avg       0.46      0.49      0.46       444



Issues to Discussion

In [43]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [44]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [45]:
df1_shuffled['concatenated'] = df1_shuffled['Title'] + ' ' + df1_shuffled['Description'] + ' ' + df1_shuffled['Comments']

Training ...

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [47]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [48]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5373134328358209

Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.52      0.53       199
           1       0.54      0.55      0.55       203

    accuracy                           0.54       402
   macro avg       0.54      0.54      0.54       402
weighted avg       0.54      0.54      0.54       402



Issues to Discussion with Description alone

In [19]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [20]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [21]:
df1_shuffled['concatenated'] = df1_shuffled['Description']

Training ...

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [23]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [24]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5074626865671642

Classification Report:
               precision    recall  f1-score   support

           0       0.50      0.47      0.48       199
           1       0.51      0.55      0.53       203

    accuracy                           0.51       402
   macro avg       0.51      0.51      0.51       402
weighted avg       0.51      0.51      0.51       402



Description+ First Comment

In [37]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

Function to extract Repository and Issue Number

In [38]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the First Comment

In [40]:
df1['Comment'] =  None
for index,row in tqdm(df1.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df1.at[index, 'Comment'] = repo_comment

2010it [14:46,  2.27it/s]


In [41]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [44]:
df1_shuffled['concatenated'] = df1_shuffled['Description'] + ' ' + str(df1_shuffled['Comment'])

Training ...

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [46]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [47]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.48507462686567165

Classification Report:
               precision    recall  f1-score   support

           0       0.48      0.43      0.45       199
           1       0.49      0.54      0.52       203

    accuracy                           0.49       402
   macro avg       0.48      0.48      0.48       402
weighted avg       0.48      0.49      0.48       402



Issues to Discussion with Comments

In [48]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [49]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [50]:
df1_shuffled['concatenated'] = df1_shuffled['Comments']

Training ...

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [52]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

Testing ...

In [53]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.49502487562189057

Classification Report:
               precision    recall  f1-score   support

           0       0.49      0.43      0.46       199
           1       0.50      0.56      0.53       203

    accuracy                           0.50       402
   macro avg       0.49      0.49      0.49       402
weighted avg       0.49      0.50      0.49       402

